# 09 · Esecuzione durevole e HITL

Coda SQLite minimale con idempotenza, lease e interrupt persistenti. Un restart viene
simulato chiudendo e riaprendo la connessione.

## Obiettivi, prerequisiti e modalità di lettura

Simulerai coda durevole, idempotenza, restart e locking con SQLite. Durata: 25–35 minuti. Tutto gira offline.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## 1 · Schema e stati terminali espliciti

### Spiegazione del blocco · Schema durevole

SQLite conserva lavoro e interrupt. Stati terminali distinti evitano l'ambiguità di un generico `completed=false`.

In [ ]:
import json, sqlite3, uuid
from pathlib import Path
from tempfile import TemporaryDirectory

TERMINAL = {"completed", "incomplete", "failed_verification", "budget_exceeded", "security_stop"}

def open_store(path: Path):
    db = sqlite3.connect(path)
    db.row_factory = sqlite3.Row
    db.executescript('''
    CREATE TABLE IF NOT EXISTS work (
        id TEXT PRIMARY KEY, idem TEXT UNIQUE, state TEXT, owner TEXT, version INTEGER, payload TEXT
    );
    CREATE TABLE IF NOT EXISTS interrupts (
        id TEXT PRIMARY KEY, run_id TEXT, status TEXT, payload TEXT, resolution TEXT
    );
    ''')
    return db

### Output atteso

Nessun output. Tabelle create quando `open_store` viene chiamata.

## 2 · Enqueue idempotente e claim con lease logico

### Spiegazione del blocco · Idempotenza e claim

`enqueue` riusa una chiave già vista; `claim` cambia stato con controllo versione. Sono i nuclei di deduplicazione e optimistic locking.

In [ ]:
def enqueue(db, idem: str, payload: dict) -> str:
    existing = db.execute("SELECT id FROM work WHERE idem=?", (idem,)).fetchone()
    if existing:
        return existing["id"]
    run_id = str(uuid.uuid4())
    db.execute("INSERT INTO work VALUES (?, ?, 'queued', NULL, 0, ?)", (run_id, idem, json.dumps(payload)))
    db.commit()
    return run_id

def claim(db, owner: str):
    row = db.execute("SELECT * FROM work WHERE state='queued' LIMIT 1").fetchone()
    if not row:
        return None
    db.execute("UPDATE work SET state='running', owner=?, version=version+1 WHERE id=? AND version=?", (owner, row["id"], row["version"]))
    db.commit()
    return db.execute("SELECT * FROM work WHERE id=?", (row["id"],)).fetchone()

### Output atteso

Nessun output. Helper definiti.

## 3 · Interrupt persistente, poi restart

### Spiegazione del blocco · Restart simulato

La connessione viene chiusa mentre esiste un interrupt pending. Alla riapertura, decisione e transizione terminale riprendono dallo storage.

In [ ]:
with TemporaryDirectory() as temporary:
    path = Path(temporary) / "runs.sqlite"
    db = open_store(path)
    one = enqueue(db, "session:request-1", {"goal": "crea report"})
    two = enqueue(db, "session:request-1", {"goal": "duplicato"})
    assert one == two
    item = claim(db, "worker-a")
    interrupt_id = str(uuid.uuid4())
    db.execute("INSERT INTO interrupts VALUES (?, ?, 'pending', ?, NULL)", (interrupt_id, item["id"], json.dumps({"command": "[redacted]"})))
    db.commit(); db.close()

    db = open_store(path)  # restart
    pending = db.execute("SELECT * FROM interrupts WHERE status='pending'").fetchone()
    assert pending["id"] == interrupt_id
    db.execute("UPDATE interrupts SET status='resolved', resolution=? WHERE id=? AND status='pending'", (json.dumps({"decision": "approve"}), interrupt_id))
    db.execute("UPDATE work SET state='completed', version=version+1 WHERE id=?", (item["id"],))
    db.commit()
    print(dict(db.execute("SELECT id, state, version FROM work").fetchone()))
    db.close()

### Output atteso

Un dizionario con id, `state='completed'` e versione incrementata.

## Perché non basta

SQLite rende durevole orchestrazione, non effetti esterni. Un provider email o deploy
deve accettare la stessa chiave idempotente; altrimenti un retry può duplicare l’effetto.

## Prova tu

Aggiungi `lease_until`; rendi reclamabile un item `running` solo dopo scadenza.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: deduplicare una richiesta

### Spiegazione del blocco

La stessa chiave idempotente deve restituire lo stesso identificatore anche se il payload del retry differisce.

In [ ]:
with TemporaryDirectory() as temporary:
    db = open_store(Path(temporary) / "idem.sqlite")
    primo = enqueue(db, "utente:richiesta-7", {"tentativo": 1})
    secondo = enqueue(db, "utente:richiesta-7", {"tentativo": 2})
    print("stesso id:", primo == secondo, primo)
    db.close()

### Output atteso

`stesso id: True` seguito da un UUID.

## Esempio aggiuntivo: optimistic locking

### Spiegazione del blocco

Due worker non devono completare lo stesso item usando la stessa versione letta in precedenza.

In [ ]:
with TemporaryDirectory() as temporary:
    db = open_store(Path(temporary) / "lock.sqlite")
    run_id = enqueue(db, "lock-demo", {})
    item = claim(db, "worker-a")
    versione = item["version"]
    prima = db.execute("UPDATE work SET state='completed', version=version+1 WHERE id=? AND version=?", (run_id, versione)).rowcount
    seconda = db.execute("UPDATE work SET state='completed', version=version+1 WHERE id=? AND version=?", (run_id, versione)).rowcount
    print("prima transizione:", prima, "seconda obsoleta:", seconda)
    db.close()

### Output atteso

`prima transizione: 1 seconda obsoleta: 0`.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.